In [5]:
from pathlib import Path
import os
import re
import gc
import time
import math
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# =========================================================
# 0. 경로 / 설정 (여기만 먼저 수정)
# =========================================================
IMG_ROOT = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/1. 원본상추 crops_filter(1)")
OUT_ROOT = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step1_core_geometry")

# 이미지 탐색
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
RECURSIVE = True

# 처리 범위 (원하면 제한)
BED_MIN = 0
BED_MAX = 99
LIMIT_IMAGES = None   # 예: 100 으로 테스트, 전체는 None

# 병렬 / chunk
MAX_WORKERS = 8
CHUNK_SIZE = 200
SAVE_EVERY_CHUNK = True

# 중심점 찾기 관련 (260314 기반용)
CENTER_BLUR_KERNEL = 9
CENTER_BIN_THRESH = 35
CENTER_MIN_CONTOUR_AREA = 300
CENTER_TOP_SEARCH_RATIO = 0.70   # 위쪽 영역만 후보로 조금 더 우선
CENTER_VIS_RADIUS = 5

# 타원 찾기 관련 (260322 안1-D 핵심 파라미터)
CANNY_LOW = 30
CANNY_HIGH = 70
MASK_BIN_THRESH = 40
OUTER_LIMIT_RATIO = 0.30
INNER_LIMIT_RATIO = 0.25   # inner_limit = outer_limit * 0.25 (원문 유지)
ANGLE_STEP = 5
ELLIPSE_MIN_POINTS = 10
ELLIPSE_VIS_THICKNESS = 2

# 저장 옵션
SAVE_OVERLAY = True
SAVE_DEBUG = False
CSV_NAME = "step1_core_geometry.csv"
LOG_NAME = "step1_core_geometry_log.txt"

In [6]:
# =========================================================
# 1. 유틸
# =========================================================
def now_str():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def format_seconds(sec):
    sec = max(0, int(sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def write_log(msg: str):
    print(msg)
    with open(OUT_ROOT / LOG_NAME, "a", encoding="utf-8") as f:
        f.write(msg + "\n")


def recursive_find_images(root: Path):
    if RECURSIVE:
        files = [p for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS]
    else:
        files = [p for p in root.glob("*") if p.suffix.lower() in IMG_EXTS]
    return sorted(files)


def extract_bed_number(path: Path):
    m = re.search(r"bed(\d{2})", str(path).lower())
    if m:
        return int(m.group(1))
    return None


def filter_bed_range(paths, bed_min=0, bed_max=99):
    out = []
    for p in paths:
        b = extract_bed_number(p)
        if b is None:
            out.append(p)
        elif bed_min <= b <= bed_max:
            out.append(p)
    return out


def chunk_list(lst, chunk_size):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]


def safe_relpath(path: Path, root: Path):
    try:
        return str(path.relative_to(root))
    except Exception:
        return path.name


def clamp(v, low, high):
    return max(low, min(high, v))

# =========================================================
# 2. 중심점 추출 (260314 기반 단순/안정 버전)
#    - 복잡한 후보 다투기보다 "상추 본체 contour + 상부 중심 우선"
# =========================================================
def find_center_point(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (CENTER_BLUR_KERNEL, CENTER_BLUR_KERNEL), 0)
    _, mask = cv2.threshold(blur, CENTER_BIN_THRESH, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = [c for c in contours if cv2.contourArea(c) >= CENTER_MIN_CONTOUR_AREA]
    if not contours:
        h, w = gray.shape
        return {
            "ok": False,
            "center_x": w // 2,
            "center_y": h // 2,
            "center_method": "fallback_image_center",
            "center_score": 0.0,
            "mask": mask,
            "main_contour": None,
            "reason": "no_contour",
        }

    main_cnt = max(contours, key=cv2.contourArea)
    M = cv2.moments(main_cnt)
    h, w = gray.shape
    if M["m00"] == 0:
        cx = w // 2
        cy = h // 2
    else:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])

    x, y, bw, bh = cv2.boundingRect(main_cnt)
    top_limit = y + int(bh * CENTER_TOP_SEARCH_RATIO)

    ys, xs = np.where(mask > 0)
    if len(xs) > 0:
        valid = ys <= top_limit
        if valid.sum() > 30:
            cx2 = int(np.median(xs[valid]))
            cy2 = int(np.median(ys[valid]))
            center_x = int(round((cx + cx2) / 2.0))
            center_y = int(round((cy + cy2) / 2.0))
            center_method = "centroid_topblend"
            center_score = 1.0
        else:
            center_x = cx
            center_y = cy
            center_method = "contour_centroid"
            center_score = 0.7
    else:
        center_x = cx
        center_y = cy
        center_method = "contour_centroid"
        center_score = 0.7

    center_x = clamp(center_x, 0, w - 1)
    center_y = clamp(center_y, 0, h - 1)

    return {
        "ok": True,
        "center_x": center_x,
        "center_y": center_y,
        "center_method": center_method,
        "center_score": center_score,
        "mask": mask,
        "main_contour": main_cnt,
        "reason": "ok",
    }

# =========================================================
# 3. 타원 추출 (260322 안1-D 메인 사용)
#    - 중심점은 260314 결과를 사용
# =========================================================
def find_ellipse_from_center(img_bgr, center_x, center_y, pre_mask=None, pre_main_cnt=None):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 0)
    edges = cv2.Canny(blurred, CANNY_LOW, CANNY_HIGH)

    if pre_mask is None:
        _, mask = cv2.threshold(gray, MASK_BIN_THRESH, 255, cv2.THRESH_BINARY)
    else:
        mask = pre_mask.copy()

    h, w = gray.shape
    cx = int(center_x)
    cy = int(center_y)

    if pre_main_cnt is None:
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return {
                "ok": False,
                "reason": "no_contour_for_ellipse",
                "edge_points": [],
                "ellipse": None,
                "mask": mask,
                "edges": edges,
            }
        main_cnt = max(contours, key=cv2.contourArea)
    else:
        main_cnt = pre_main_cnt

    x_m, y_m, w_m, h_m = cv2.boundingRect(main_cnt)
    outer_limit = int(max(w_m, h_m) * OUTER_LIMIT_RATIO)
    inner_limit = int(outer_limit * INNER_LIMIT_RATIO)

    edge_points = []

    for angle in range(0, 360, ANGLE_STEP):
        rad = np.deg2rad(angle)
        possible_points = []

        for r in range(inner_limit, outer_limit):
            curr_x = int(cx + r * np.cos(rad))
            curr_y = int(cy + r * np.sin(rad))
            if 0 <= curr_x < w and 0 <= curr_y < h:
                if edges[curr_y, curr_x] > 0 and mask[curr_y, curr_x] > 0:
                    possible_points.append([curr_x, curr_y])

        if possible_points:
            edge_points.append(possible_points[-1])  # 안1-D 원문 유지
        else:
            for r in range(outer_limit, inner_limit, -1):
                tx = int(cx + r * np.cos(rad))
                ty = int(cy + r * np.sin(rad))
                if 0 <= tx < w and 0 <= ty < h and mask[ty, tx] > 0:
                    edge_points.append([tx, ty])
                    break

    if len(edge_points) < ELLIPSE_MIN_POINTS:
        return {
            "ok": False,
            "reason": "not_enough_edge_points",
            "edge_points": edge_points,
            "ellipse": None,
            "mask": mask,
            "edges": edges,
        }

    pts = np.array(edge_points, dtype=np.int32)

    try:
        ellipse = cv2.fitEllipse(pts)
    except Exception:
        return {
            "ok": False,
            "reason": "fitEllipse_failed",
            "edge_points": edge_points,
            "ellipse": None,
            "mask": mask,
            "edges": edges,
        }

    return {
        "ok": True,
        "reason": "ok",
        "edge_points": edge_points,
        "ellipse": ellipse,
        "mask": mask,
        "edges": edges,
        "outer_limit": outer_limit,
        "inner_limit": inner_limit,
    }

# =========================================================
# 4. 시각화
# =========================================================
def draw_overlay(img_bgr, center_info, ellipse_info):
    vis = img_bgr.copy()

    cx = int(center_info["center_x"])
    cy = int(center_info["center_y"])
    cv2.circle(vis, (cx, cy), CENTER_VIS_RADIUS, (0, 140, 255), -1)  # 주황 중심점

    if ellipse_info["ok"] and ellipse_info["ellipse"] is not None:
        ellipse = ellipse_info["ellipse"]
        cv2.ellipse(vis, ellipse, (255, 0, 0), ELLIPSE_VIS_THICKNESS)  # 파란 타원

        for p in ellipse_info["edge_points"]:
            cv2.circle(vis, tuple(map(int, p)), 1, (0, 255, 255), -1)

    return vis

# =========================================================
# 5. 단일 이미지 처리
# =========================================================
def process_one_image(img_path_str):
    img_path = Path(img_path_str)
    rel_path = safe_relpath(img_path, IMG_ROOT)

    try:
        img = cv2.imread(str(img_path))
        if img is None:
            return {
                "image_path": str(img_path),
                "rel_path": rel_path,
                "status": "fail",
                "fail_reason": "imread_failed",
            }

        h, w = img.shape[:2]

        center_info = find_center_point(img)
        ellipse_info = find_ellipse_from_center(
            img_bgr=img,
            center_x=center_info["center_x"],
            center_y=center_info["center_y"],
            pre_mask=center_info.get("mask"),
            pre_main_cnt=center_info.get("main_contour"),
        )

        overlay_rel = None
        debug_mask_rel = None
        debug_edge_rel = None

        if SAVE_OVERLAY:
            overlay = draw_overlay(img, center_info, ellipse_info)
            save_path = OUT_ROOT / "overlay" / rel_path
            ensure_dir(save_path.parent)
            cv2.imwrite(str(save_path), overlay)
            overlay_rel = safe_relpath(save_path, OUT_ROOT)

        if SAVE_DEBUG:
            mask_path = OUT_ROOT / "debug_mask" / rel_path
            edge_path = OUT_ROOT / "debug_edge" / rel_path
            ensure_dir(mask_path.parent)
            ensure_dir(edge_path.parent)
            cv2.imwrite(str(mask_path), center_info["mask"])
            cv2.imwrite(str(edge_path), ellipse_info["edges"])
            debug_mask_rel = safe_relpath(mask_path, OUT_ROOT)
            debug_edge_rel = safe_relpath(edge_path, OUT_ROOT)

        row = {
            "image_path": str(img_path),
            "rel_path": rel_path,
            "image_name": img_path.name,
            "stem": img_path.stem,
            "width": w,
            "height": h,
            "status": "ok" if ellipse_info["ok"] else "partial",
            "fail_reason": ellipse_info["reason"] if not ellipse_info["ok"] else "",
            "center_x": int(center_info["center_x"]),
            "center_y": int(center_info["center_y"]),
            "center_method": center_info["center_method"],
            "center_score": float(center_info["center_score"]),
            "overlay_path": overlay_rel,
            "debug_mask_path": debug_mask_rel,
            "debug_edge_path": debug_edge_rel,
        }

        if ellipse_info["ok"] and ellipse_info["ellipse"] is not None:
            (e_cx, e_cy), (axis1, axis2), angle_deg = ellipse_info["ellipse"]
            major_axis = float(max(axis1, axis2))
            minor_axis = float(min(axis1, axis2))
            axis_ratio = major_axis / minor_axis if minor_axis > 0 else np.nan

            row.update({
                "ellipse_cx": float(e_cx),
                "ellipse_cy": float(e_cy),
                "major_axis": major_axis,
                "minor_axis": minor_axis,
                "axis_ratio": float(axis_ratio) if not np.isnan(axis_ratio) else np.nan,
                "angle_deg": float(angle_deg),
                "edge_point_n": int(len(ellipse_info["edge_points"])),
                "inner_limit": int(ellipse_info.get("inner_limit", -1)),
                "outer_limit": int(ellipse_info.get("outer_limit", -1)),
                "ellipse_ok": 1,
            })
        else:
            row.update({
                "ellipse_cx": np.nan,
                "ellipse_cy": np.nan,
                "major_axis": np.nan,
                "minor_axis": np.nan,
                "axis_ratio": np.nan,
                "angle_deg": np.nan,
                "edge_point_n": int(len(ellipse_info.get("edge_points", []))),
                "inner_limit": int(ellipse_info.get("inner_limit", -1)) if isinstance(ellipse_info, dict) else -1,
                "outer_limit": int(ellipse_info.get("outer_limit", -1)) if isinstance(ellipse_info, dict) else -1,
                "ellipse_ok": 0,
            })

        return row

    except Exception as e:
        return {
            "image_path": str(img_path),
            "rel_path": rel_path,
            "status": "fail",
            "fail_reason": f"exception: {type(e).__name__}: {e}",
        }

# =========================================================
# 6. chunk 저장
# =========================================================
def save_chunk_rows(rows, chunk_idx):
    if not rows:
        return None
    chunk_dir = OUT_ROOT / "chunks"
    ensure_dir(chunk_dir)
    chunk_path = chunk_dir / f"chunk_{chunk_idx:03d}.csv"
    pd.DataFrame(rows).to_csv(chunk_path, index=False, encoding="utf-8-sig")
    return chunk_path



In [7]:
# =========================================================
# 7. 메인
# =========================================================
def main():
    ensure_dir(OUT_ROOT)
    ensure_dir(OUT_ROOT / "overlay")
    if SAVE_DEBUG:
        ensure_dir(OUT_ROOT / "debug_mask")
        ensure_dir(OUT_ROOT / "debug_edge")

    with open(OUT_ROOT / LOG_NAME, "w", encoding="utf-8") as f:
        f.write("")

    start_time = time.time()
    write_log("=" * 90)
    write_log(f"[START] {now_str()}")
    write_log(f"IMG_ROOT      : {IMG_ROOT}")
    write_log(f"OUT_ROOT      : {OUT_ROOT}")
    write_log(f"MAX_WORKERS   : {MAX_WORKERS}")
    write_log(f"CHUNK_SIZE    : {CHUNK_SIZE}")
    write_log(f"BED_RANGE     : bed{BED_MIN:02d} ~ bed{BED_MAX:02d}")

    all_imgs = recursive_find_images(IMG_ROOT)
    all_imgs = filter_bed_range(all_imgs, BED_MIN, BED_MAX)

    if LIMIT_IMAGES is not None:
        all_imgs = all_imgs[:LIMIT_IMAGES]

    total_n = len(all_imgs)
    write_log(f"TOTAL_IMAGES  : {total_n}")

    if total_n == 0:
        write_log("[STOP] 이미지가 없습니다.")
        return

    all_rows = []
    total_done = 0
    chunk_idx = 0

    for paths in chunk_list(all_imgs, CHUNK_SIZE):
        chunk_idx += 1
        chunk_start = time.time()
        rows = []

        write_log("-" * 90)
        write_log(f"[CHUNK {chunk_idx}] start / size={len(paths)} / done={total_done}/{total_n}")

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {ex.submit(process_one_image, str(p)): p for p in paths}
            for fut in tqdm(as_completed(futures), total=len(futures), desc=f"chunk {chunk_idx}"):
                row = fut.result()
                rows.append(row)
                total_done += 1

        rows = pd.DataFrame(rows)
        if len(rows) > 0:
            rows = rows.sort_values("rel_path").reset_index(drop=True)
            rows_list = rows.to_dict(orient="records")
        else:
            rows_list = []

        all_rows.extend(rows_list)

        if SAVE_EVERY_CHUNK:
            chunk_path = save_chunk_rows(rows_list, chunk_idx)
            write_log(f"[CHUNK {chunk_idx}] saved: {chunk_path}")

        ok_n = int((rows["ellipse_ok"] == 1).sum()) if "ellipse_ok" in rows.columns else 0
        fail_n = int((rows["status"] == "fail").sum()) if "status" in rows.columns else 0
        partial_n = int((rows["status"] == "partial").sum()) if "status" in rows.columns else 0
        chunk_elapsed = time.time() - chunk_start
        total_elapsed = time.time() - start_time
        eta = (total_elapsed / total_done) * (total_n - total_done) if total_done > 0 else 0

        write_log(
            f"[CHUNK {chunk_idx}] ok={ok_n}, partial={partial_n}, fail={fail_n}, "
            f"chunk_time={format_seconds(chunk_elapsed)}, total_done={total_done}/{total_n}, ETA={format_seconds(eta)}"
        )

        del rows, rows_list
        gc.collect()

    final_df = pd.DataFrame(all_rows)
    final_df = final_df.sort_values("rel_path").reset_index(drop=True)
    final_csv_path = OUT_ROOT / CSV_NAME
    final_df.to_csv(final_csv_path, index=False, encoding="utf-8-sig")

    ok_n = int((final_df["ellipse_ok"] == 1).sum()) if "ellipse_ok" in final_df.columns else 0
    partial_n = int((final_df["status"] == "partial").sum()) if "status" in final_df.columns else 0
    fail_n = int((final_df["status"] == "fail").sum()) if "status" in final_df.columns else 0

    fail_summary_path = OUT_ROOT / "fail_reason_summary.csv"
    if "fail_reason" in final_df.columns:
        final_df["fail_reason"].fillna("ok").value_counts().rename_axis("fail_reason").reset_index(name="count").to_csv(
            fail_summary_path, index=False, encoding="utf-8-sig"
        )

    write_log("=" * 90)
    write_log(f"[FINAL CSV] {final_csv_path}")
    write_log(f"[FAIL SUMMARY] {fail_summary_path}")
    write_log(f"[RESULT] ok={ok_n}, partial={partial_n}, fail={fail_n}, total={len(final_df)}")
    write_log(f"[END] {now_str()} / elapsed={format_seconds(time.time() - start_time)}")
    write_log("=" * 90)



In [8]:

if __name__ == "__main__":
    main()


[START] 2026-03-29 06:50:01
IMG_ROOT      : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/1. 원본상추 crops_filter(1)
OUT_ROOT      : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step1_core_geometry
MAX_WORKERS   : 8
CHUNK_SIZE    : 200
BED_RANGE     : bed00 ~ bed99
TOTAL_IMAGES  : 1088
------------------------------------------------------------------------------------------
[CHUNK 1] start / size=200 / done=0/1088


chunk 1:   0%|          | 0/200 [00:00<?, ?it/s]

[CHUNK 1] saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step1_core_geometry/chunks/chunk_001.csv
[CHUNK 1] ok=200, partial=0, fail=0, chunk_time=00:00:31, total_done=200/1088, ETA=00:02:36
------------------------------------------------------------------------------------------
[CHUNK 2] start / size=200 / done=200/1088


chunk 2:   0%|          | 0/200 [00:00<?, ?it/s]

[CHUNK 2] saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step1_core_geometry/chunks/chunk_002.csv
[CHUNK 2] ok=200, partial=0, fail=0, chunk_time=00:00:41, total_done=400/1088, ETA=00:02:12
------------------------------------------------------------------------------------------
[CHUNK 3] start / size=200 / done=400/1088


...

[CHUNK 4] saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step1_core_geometry/chunks/chunk_004.csv
[CHUNK 4] ok=200, partial=0, fail=0, chunk_time=00:00:38, total_done=800/1088, ETA=00:00:56
------------------------------------------------------------------------------------------
[CHUNK 5] start / size=200 / done=800/1088


chunk 5:   0%|          | 0/200 [00:00<?, ?it/s]

[CHUNK 5] saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step1_core_geometry/chunks/chunk_005.csv
[CHUNK 5] ok=200, partial=0, fail=0, chunk_time=00:00:38, total_done=1000/1088, ETA=00:00:17
------------------------------------------------------------------------------------------
[CHUNK 6] start / size=88 / done=1000/1088


chunk 6:   0%|          | 0/88 [00:00<?, ?it/s]

[CHUNK 6] saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step1_core_geometry/chunks/chunk_006.csv
[CHUNK 6] ok=88, partial=0, fail=0, chunk_time=00:00:19, total_done=1088/1088, ETA=00:00:00
[FINAL CSV] /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step1_core_geometry/step1_core_geometry.csv
[FAIL SUMMARY] /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step1_core_geometry/fail_reason_summary.csv
[RESULT] ok=1088, partial=0, fail=0, total=1088
[END] 2026-03-29 06:53:37 / elapsed=00:03:36
